In [63]:
# @title ⬇️ Download do Boletim de Urna (BUWeb) oficial do TSE
from pathlib import Path
import re
import zipfile

import pandas as pd
import requests

# Use UMA das opções abaixo:
# 1) informe o link completo copiado do TSE em URL_BU; ou
# 2) deixe URL_BU vazio e informe ANO, UF, TURNO e GERACAO_BU.
ANO = 2026
UF = "RN"
TURNO = 2
GERACAO_BU = "261020260742"  # pode ficar vazio se URL_BU for informado
URL_BU = ""  # exemplo: https://cdn.tse.jus.br/.../bweb_2t_RN_261020260742.zip

BASE_DIR = Path.cwd()
DADOS_DIR = BASE_DIR / "planilhas"
DOWNLOADS_DIR = DADOS_DIR / "downloads"
DADOS_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_ORIGINAL = "latin-1"


def url_buweb_tse(ano, turno, uf, geracao):
    """Monta o link oficial do ZIP de Boletins de Urna do TSE."""
    return (
        f"https://cdn.tse.jus.br/estatistica/sead/eleicoes/"
        f"eleicoes{ano}/buweb/bweb_{turno}t_{uf}_{geracao}.zip"
    )


UF = UF.strip().upper()
GERACAO_BU = str(GERACAO_BU).strip()
URL_BU = str(URL_BU).strip()
if URL_BU:
    if not URL_BU.lower().startswith("https://cdn.tse.jus.br/"):
        raise ValueError("URL_BU deve apontar para o CDN oficial https://cdn.tse.jus.br/.")
    url_tse = URL_BU
    nome_zip_url = Path(url_tse.split("?", 1)[0]).name
    padrao_nome = re.fullmatch(r"bweb_([12])t_([A-Z]{2})_(\d{12})\.zip", nome_zip_url, re.I)
    padrao_ano = re.search(r"/eleicoes(\d{4})/", url_tse, re.I)
    if not padrao_nome or not padrao_ano:
        raise ValueError(
            "URL_BU deve seguir o padrão do BUWeb, por exemplo "
            "bweb_2t_RN_261020260742.zip."
        )
    TURNO = int(padrao_nome.group(1))
    UF = padrao_nome.group(2).upper()
    GERACAO_BU = padrao_nome.group(3)
    ANO = int(padrao_ano.group(1))
else:
    if not re.fullmatch(r"[A-Z]{2}", UF):
        raise ValueError("UF deve ser uma sigla de dois caracteres, por exemplo RN ou SP.")
    if TURNO not in (1, 2):
        raise ValueError("TURNO deve ser 1 ou 2.")
    if not isinstance(ANO, int) or not 2000 <= ANO <= 2100:
        raise ValueError("ANO deve ser um ano eleitoral válido, como 2026.")
    if not re.fullmatch(r"\d{12}", GERACAO_BU):
        raise ValueError(
            "Sem URL_BU, GERACAO_BU deve conter os 12 dígitos do nome exato do ZIP do TSE."
        )
    url_tse = url_buweb_tse(ANO, TURNO, UF, GERACAO_BU)

nome_zip = Path(url_tse.split("?", 1)[0]).name
if not nome_zip.lower().endswith(".zip"):
    raise ValueError("URL_BU deve apontar para um arquivo .zip do BUWeb.")
zip_path = DOWNLOADS_DIR / nome_zip
ARQUIVO_SAIDA = DADOS_DIR / f"dados_eleitorais_{UF}_{TURNO}T_{ANO}_enxuto.csv"

if not zip_path.exists():
    print(f"Baixando Boletins de Urna oficiais do TSE: {url_tse}")
    with requests.get(url_tse, stream=True, timeout=120) as resposta:
        resposta.raise_for_status()
        with zip_path.open("wb") as arquivo:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    arquivo.write(bloco)
else:
    print(f"ZIP já baixado: {zip_path}")

with zipfile.ZipFile(zip_path) as compactado:
    csvs = [nome for nome in compactado.namelist() if nome.lower().endswith(".csv")]
    if not csvs:
        raise RuntimeError(f"O ZIP do TSE não contém CSV: {zip_path}")
    csv_path = DADOS_DIR / Path(csvs[0]).name
    if not csv_path.exists():
        with compactado.open(csvs[0]) as origem, csv_path.open("wb") as destino:
            destino.write(origem.read())

ARQUIVO_ORIGINAL = csv_path
print(f"Fonte oficial: {url_tse}")
print(f"Arquivo extraído: {ARQUIVO_ORIGINAL}")

colunas = pd.read_csv(
    ARQUIVO_ORIGINAL,
    sep=SEPARADOR,
    encoding=ENCODING_ORIGINAL,
    dtype=str,
    keep_default_na=False,
    nrows=0,
).columns
colunas_esperadas = {
    "DS_CARGO_PERGUNTA", "DS_TIPO_VOTAVEL", "QT_VOTOS",
    "QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES",
    "QT_ELEI_BIOM_SEM_HABILITACAO",
}
faltando = sorted(colunas_esperadas.difference(colunas))
if faltando:
    raise ValueError(f"O arquivo não tem o layout de BU esperado. Colunas ausentes: {faltando}")

print("Layout de BU validado: Governador/Presidente, tipos de voto e dados eleitorais disponíveis.")

ZIP já baixado: /content/planilhas/downloads/bweb_2t_RN_261020260742.zip
Fonte oficial: https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/buweb/bweb_2t_RN_261020260742.zip
Arquivo extraído: /content/planilhas/bweb_2t_RN_261020260742.csv
Layout de BU validado: Governador/Presidente, tipos de voto e dados eleitorais disponíveis.


In [64]:
# @title 💾 Confirmação e download do CSV original
# O CSV original já foi extraído e salvo na célula de download.
print(f"CSV original salvo em: {ARQUIVO_ORIGINAL.resolve()}")
print(f"Tamanho do arquivo: {ARQUIVO_ORIGINAL.stat().st_size / 1024 / 1024:.2f} MB")

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

if colab_files is not None:
    colab_files.download(str(ARQUIVO_ORIGINAL))
else:
    print("No Jupyter local, o arquivo já está disponível no caminho acima.")

CSV original salvo em: /content/planilhas/bweb_2t_RN_261020260742.csv
Tamanho do arquivo: 26.88 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
# @title 🔎 Inspeção dos dados baixados
# Exibe a estrutura do CSV e amostras dos cargos trabalhados na análise.
print(f"Arquivo inspecionado: {ARQUIVO_ORIGINAL}")

# Lê apenas uma amostra para exibir o cabeçalho e os primeiros registros.
amostra = pd.read_csv(
    ARQUIVO_ORIGINAL,
    sep=SEPARADOR,
    encoding=ENCODING_ORIGINAL,
    dtype=str,
    keep_default_na=False,
    nrows=5,
)

print(f"\nNúmero de colunas: {len(amostra.columns)}")
print("Cabeçalho:")
print(list(amostra.columns))

# Conta todas as linhas sem carregar o arquivo completo na memória.
linhas_totais = 0
for bloco in pd.read_csv(
    ARQUIVO_ORIGINAL,
    sep=SEPARADOR,
    encoding=ENCODING_ORIGINAL,
    dtype=str,
    keep_default_na=False,
    usecols=[amostra.columns[0]],
    chunksize=200_000,
    low_memory=False,
):
    linhas_totais += len(bloco)

print(f"\nNúmero de linhas: {linhas_totais:,}")
print("\nDez primeiros registros do arquivo:")
display(amostra)

# O arquivo de 2026 usa DS_CARGO; versões anteriores podem usar
# DS_CARGO_PERGUNTA. Mostra uma amostra específica dos cargos analisados.
coluna_cargo = next(
    (coluna for coluna in ("DS_CARGO_PERGUNTA", "DS_CARGO") if coluna in amostra.columns),
    None,
)
if coluna_cargo:
    cargos_alvo = {"Governador", "Presidente"}
    encontrados = []
    for bloco in pd.read_csv(
        ARQUIVO_ORIGINAL,
        sep=SEPARADOR,
        encoding=ENCODING_ORIGINAL,
        dtype=str,
        keep_default_na=False,
        usecols=lambda coluna: coluna in {coluna_cargo, "NR_TURNO"},
        chunksize=200_000,
        low_memory=False,
    ):
        filtro = bloco[coluna_cargo].isin(cargos_alvo)
        if "NR_TURNO" in bloco.columns:
            filtro &= bloco["NR_TURNO"].astype(str).str.strip().eq(str(TURNO))
        encontrados.append(bloco.loc[filtro].head(5))
        if sum(len(item) for item in encontrados) >= 10:
            break

    cargos_amostra = pd.concat(encontrados, ignore_index=True).head(5) if encontrados else pd.DataFrame()
    print(f"\nAmostra de Governador/Presidente ({coluna_cargo}):")
    display(cargos_amostra)
else:
    print("\nA coluna de cargo não foi encontrada no arquivo baixado.")

Arquivo inspecionado: /content/planilhas/bweb_2t_RN_261020260742.csv

Número de colunas: 45
Cabeçalho:
['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_PLEITO', 'DT_PLEITO', 'NR_TURNO', 'CD_ELEICAO', 'DS_ELEICAO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'NR_LOCAL_VOTACAO', 'CD_CARGO_PERGUNTA', 'DS_CARGO_PERGUNTA', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'DT_BU_RECEBIDO', 'QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCOES', 'CD_TIPO_URNA', 'DS_TIPO_URNA', 'CD_TIPO_VOTAVEL', 'DS_TIPO_VOTAVEL', 'NR_VOTAVEL', 'NM_VOTAVEL', 'QT_VOTOS', 'NR_URNA_EFETIVADA', 'CD_CARGA_1_URNA_EFETIVADA', 'CD_CARGA_2_URNA_EFETIVADA', 'CD_FLASHCARD_URNA_EFETIVADA', 'DT_CARGA_URNA_EFETIVADA', 'DS_CARGO_PERGUNTA_SECAO', 'DS_SECOES_AGREGADAS', 'DT_ABERTURA', 'DT_ENCERRAMENTO', 'QT_ELEI_BIOM_SEM_HABILITACAO', 'DT_EMISSAO_BU', 'NR_JUNTA_APURADORA', 'NR_TURMA_APURADORA']

Número de linhas: 63,676

Dez primeiros registros do arquivo:


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,...,CD_FLASHCARD_URNA_EFETIVADA,DT_CARGA_URNA_EFETIVADA,DS_CARGO_PERGUNTA_SECAO,DS_SECOES_AGREGADAS,DT_ABERTURA,DT_ENCERRAMENTO,QT_ELEI_BIOM_SEM_HABILITACAO,DT_EMISSAO_BU,NR_JUNTA_APURADORA,NR_TURMA_APURADORA
0,26/10/2026,07:42:38,2026,0,Eleição Ordinária,3221,2026-10-25 00:00:00,2,6260,Eleições Gerais Estaduais 2026,...,826A025A,2026-09-23 15:11:00,3 - 30,#NULO#,2026-10-25 08:00:01,2026-10-25 17:01:17,28,2026-10-25 17:03:52,-1,-1
1,26/10/2026,07:42:38,2026,0,Eleição Ordinária,3221,2026-10-25 00:00:00,2,6260,Eleições Gerais Estaduais 2026,...,826A025A,2026-09-23 17:02:00,3 - 50,#NULO#,2026-10-25 08:00:01,2026-10-25 17:03:02,32,2026-10-25 17:05:24,-1,-1
2,26/10/2026,07:42:38,2026,0,Eleição Ordinária,3221,2026-10-25 00:00:00,2,6260,Eleições Gerais Estaduais 2026,...,826A025A,2026-09-23 17:08:00,3 - 52,#NULO#,2026-10-25 08:00:01,2026-10-25 17:00:29,23,2026-10-25 17:02:25,-1,-1
3,26/10/2026,07:42:38,2026,0,Eleição Ordinária,3221,2026-10-25 00:00:00,2,6260,Eleições Gerais Estaduais 2026,...,826A025A,2026-09-23 15:52:00,3 - 40,#NULO#,2026-10-25 08:00:01,2026-10-25 17:01:13,24,2026-10-25 17:03:15,-1,-1
4,26/10/2026,07:42:38,2026,0,Eleição Ordinária,3221,2026-10-25 00:00:00,2,6260,Eleições Gerais Estaduais 2026,...,826A025A,2026-09-23 16:51:00,3 - 44,#NULO#,2026-10-25 08:00:01,2026-10-25 17:01:42,21,2026-10-25 17:05:41,-1,-1



Amostra de Governador/Presidente (DS_CARGO_PERGUNTA):


,NR_TURNO,DS_CARGO_PERGUNTA
0,2,Governador
1,2,Governador
2,2,Governador
3,2,Governador
4,2,Governador


# Célula 1 — Download, inspeção e preparação dos dados

Na primeira célula, escolha `ANO`, `UF`, `TURNO` e `GERACAO_BU` ou informe `URL_BU`. O notebook baixa o ZIP oficial de **Boletins de Urna (BUWeb)** do TSE, extrai o CSV e valida o layout que contém Governador, Presidente, tipos de voto, comparecimento, abstenções e biometria.

A célula seguinte exibe o cabeçalho, o número de colunas e linhas e uma amostra dos dados. Depois, execute a célula de confirmação para verificar o caminho do CSV original e baixá-lo para o computador quando estiver no Colab. A análise usa o arquivo original sem remover colunas.

In [66]:
# @title 🧰 Preparação: confere o Ollama, o servidor e o modelo
import os, shutil, subprocess, sys, time, requests

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen2.5:3b"
IN_COLAB = "google.colab" in sys.modules
sh = get_ipython().system

def server_is_up() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

# 1 · O programa ollama existe?
if shutil.which("ollama"):
    print("✅ Ollama já instalado")
elif IN_COLAB:
    print("⏳ instalando o Ollama (cerca de 1 minuto)...")
    sh("apt-get install -y -qq zstd > /dev/null 2>&1 || "
       "(apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)")
    sh("curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2")
    if not shutil.which("ollama"):
        raise RuntimeError("a instalação falhou. Rode a célula de novo.")
    print("✅ Ollama instalado")
else:
    raise RuntimeError("Ollama não encontrado. Instale em https://ollama.com/download")

# 2 · O servidor está de pé?
if server_is_up():
    print("✅ servidor já está de pé")
else:
    print("⏳ subindo o servidor...")
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)
    if not server_is_up():
        raise RuntimeError("o servidor não respondeu. Veja ollama.log")
    print("✅ servidor de pé")

# 3 · O modelo já foi baixado?
installed = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
if MODEL in installed:
    print(f"✅ {MODEL} já baixado")
else:
    print(f"⏳ baixando {MODEL} (~2 GB)...")
    sh(f"ollama pull {MODEL}")
    print(f"✅ {MODEL} baixado")

version = requests.get(f"{OLLAMA_URL}/api/version").json()["version"]
print(f"\n🚀 pronto · Ollama {version} · modelo {MODEL}")

✅ Ollama já instalado
✅ servidor já está de pé
✅ qwen2.5:3b já baixado

🚀 pronto · Ollama 0.40.14 · modelo qwen2.5:3b


In [67]:
# @title ⚡ Diagnóstico de GPU e velocidade do Ollama
import os
import shutil
import subprocess

# 1. GPU visível no runtime
if shutil.which("nvidia-smi"):
    gpu = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    if gpu.returncode == 0 and gpu.stdout.strip():
        print(gpu.stdout.strip())
    else:
        print("⚠️ nvidia-smi encontrou um problema ao consultar a GPU.")
else:
    print("ℹ️ nvidia-smi não está instalado; GPU NVIDIA não disponível neste ambiente.")

# 2. O Ollama está usando CUDA?
log_path = "ollama.log"
if shutil.which("grep") and os.path.exists(log_path):
    log = subprocess.run(
        ["grep", "-E", "library=CUDA|library=cpu", log_path],
        capture_output=True,
        text=True,
        check=False,
    ).stdout
    print("CUDA detectado" if "library=CUDA" in log else "⚠️ CPU — GPU não está sendo usada")
else:
    print("ℹ️ Log do Ollama não disponível para verificar CUDA.")

# 3. Teste real de velocidade
resposta = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={
        "model": MODEL,
        "prompt": "Conte de 1 a 50.",
        "stream": False,
        "options": {"num_predict": 100},
    },
    timeout=300,
)
resposta.raise_for_status()
r = resposta.json()

if r.get("eval_count") is None or not r.get("eval_duration"):
    raise RuntimeError(f"O Ollama não retornou métricas de avaliação: {r}")

velocidade = r["eval_count"] / (r["eval_duration"] / 1e9)
print(f"Velocidade: {velocidade:.1f} tokens/s")

ℹ️ nvidia-smi não está instalado; GPU NVIDIA não disponível neste ambiente.
⚠️ CPU — GPU não está sendo usada
Velocidade: 33.6 tokens/s


# Célula 2 — Utilitários

In [68]:
# @title 🔧 Utilitários: chamada ao modelo e impressão da conversa
import json, re
import pandas as pd

def llm(messages, tools=None):
    """Uma chamada ao /api/chat do Ollama. Devolve o JSON inteiro."""
    body = {"model": MODEL, "messages": messages, "stream": False,
            "options": {"temperature": 0, "num_predict": 600}}
    if tools:
        body["tools"] = tools
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=body, timeout=300)
    r.raise_for_status()
    return r.json()

def show(obj, limit=1600):
    """Imprime um dicionário como JSON legível, cortado se for longo."""
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    print(text if len(text) <= limit else text[:limit] + "\n ... (cortado)")

ICON = {"system": "⚙️ ", "user": "👤", "assistant": "🤖", "tool": "🔧"}

def show_messages(messages):
    """A conversa inteira, uma linha por mensagem."""
    for i, m in enumerate(messages):
        if m.get("tool_calls"):
            what = "pede " + " e ".join(
                f'{c["function"]["name"]}({json.dumps(c["function"]["arguments"], ensure_ascii=False)})'
                for c in m["tool_calls"])
        else:
            what = (m.get("content") or "").replace("\n", " ")
        print(f"[{i}] {ICON[m['role']]} {m['role']:<9} │ {what[:120]}{'…' if len(what) > 120 else ''}")

print("utilitários prontos")

utilitários prontos


In [69]:
# @title 📊 Parte 1 · O gabarito: respondemos à mão, com pandas
# Usa diretamente o CSV original baixado do TSE.
if not ARQUIVO_ORIGINAL.exists():
    raise FileNotFoundError(
        f"CSV original não encontrado: {ARQUIVO_ORIGINAL}. "
        "Execute primeiro a célula de download."
    )

CSV = ARQUIVO_ORIGINAL

# dtype=str preserva zeros à esquerda em NR_ZONA, NR_SECAO e CD_MUNICIPIO.
bu = pd.read_csv(
    CSV,
    sep=";",
    encoding=ENCODING_ORIGINAL,
    dtype=str,
    keep_default_na=False,
)

# Padroniza os nomes e cria as colunas derivadas usadas pelas ferramentas.
bu = bu.rename(columns={"NR_ZONA": "ZONA_N", "NR_SECAO": "SECAO_N"})
bu["QT_VOTOS_N"] = pd.to_numeric(bu["QT_VOTOS"], errors="coerce").fillna(0)
bu["AGREGADA"] = bu["DS_SECOES_AGREGADAS"] != "#NULO#"

print(f"{len(bu)} linhas · {len(bu.columns)} colunas · {UF}/{ANO} · {TURNO}º turno\n")
print("Colunas:", ", ".join(bu.columns))

63676 linhas · 47 colunas · RN/2026 · 2º turno

Colunas: DT_GERACAO, HH_GERACAO, ANO_ELEICAO, CD_TIPO_ELEICAO, NM_TIPO_ELEICAO, CD_PLEITO, DT_PLEITO, NR_TURNO, CD_ELEICAO, DS_ELEICAO, SG_UF, CD_MUNICIPIO, NM_MUNICIPIO, ZONA_N, SECAO_N, NR_LOCAL_VOTACAO, CD_CARGO_PERGUNTA, DS_CARGO_PERGUNTA, NR_PARTIDO, SG_PARTIDO, NM_PARTIDO, DT_BU_RECEBIDO, QT_APTOS, QT_COMPARECIMENTO, QT_ABSTENCOES, CD_TIPO_URNA, DS_TIPO_URNA, CD_TIPO_VOTAVEL, DS_TIPO_VOTAVEL, NR_VOTAVEL, NM_VOTAVEL, QT_VOTOS, NR_URNA_EFETIVADA, CD_CARGA_1_URNA_EFETIVADA, CD_CARGA_2_URNA_EFETIVADA, CD_FLASHCARD_URNA_EFETIVADA, DT_CARGA_URNA_EFETIVADA, DS_CARGO_PERGUNTA_SECAO, DS_SECOES_AGREGADAS, DT_ABERTURA, DT_ENCERRAMENTO, QT_ELEI_BIOM_SEM_HABILITACAO, DT_EMISSAO_BU, NR_JUNTA_APURADORA, NR_TURMA_APURADORA, QT_VOTOS_N, AGREGADA


In [70]:
# Célula duplicada removida: o carregamento oficial está na célula anterior.
print("Carregamento já realizado na célula anterior.")

63676 linhas · 45 colunas · RN/2026 · 2º turno

Colunas: DT_GERACAO, HH_GERACAO, ANO_ELEICAO, CD_TIPO_ELEICAO, NM_TIPO_ELEICAO, CD_PLEITO, DT_PLEITO, NR_TURNO, CD_ELEICAO, DS_ELEICAO, SG_UF, CD_MUNICIPIO, NM_MUNICIPIO, ZONA_N, SECAO_N, NR_LOCAL_VOTACAO, CD_CARGO_PERGUNTA, DS_CARGO_PERGUNTA, NR_PARTIDO, SG_PARTIDO, NM_PARTIDO, DT_BU_RECEBIDO, QT_APTOS, QT_COMPARECIMENTO, QT_ABSTENCOES, CD_TIPO_URNA, DS_TIPO_URNA, CD_TIPO_VOTAVEL, DS_TIPO_VOTAVEL, NR_VOTAVEL, NM_VOTAVEL, QT_VOTOS, NR_URNA_EFETIVADA, CD_CARGA_1_URNA_EFETIVADA, CD_CARGA_2_URNA_EFETIVADA, CD_FLASHCARD_URNA_EFETIVADA, DT_CARGA_URNA_EFETIVADA, DS_CARGO_PERGUNTA_SECAO, DS_SECOES_AGREGADAS, DT_ABERTURA, DT_ENCERRAMENTO, QT_ELEI_BIOM_SEM_HABILITACAO, DT_EMISSAO_BU, NR_JUNTA_APURADORA, NR_TURMA_APURADORA


# Célula 4 — Parte 2: as ferramentas são funções Python comuns

In [71]:
def _parse_int(valor, nome):
    """Converte para int com mensagem de erro clara."""
    try:
        return int(valor), None
    except (TypeError, ValueError):
        return None, {"error": f"{nome} '{valor}' inválido. Use um número inteiro."}


def _num(v, default=0):
    """Converte para número com fallback seguro (NaN → default)."""
    n = pd.to_numeric(v, errors="coerce")
    return default if pd.isna(n) else n


def total_por_tipo(tipo_voto=None, cargo=None):
    """Total de votos por tipo, opcionalmente filtrado por cargo."""
    if tipo_voto is not None and tipo_voto not in ("Nulo", "Branco", "Nominal"):
        return {"error": f"tipo_voto '{tipo_voto}' inválido. Use 'Nulo', 'Branco' ou 'Nominal'."}
    if cargo is not None and cargo not in ("Governador", "Presidente"):
        return {"error": f"cargo '{cargo}' inválido. Use 'Governador' ou 'Presidente'."}

    r = bu.copy()
    r["QT_VOTOS_N"] = pd.to_numeric(r["QT_VOTOS"], errors="coerce").fillna(0)
    if tipo_voto is not None:
        r = r[r.DS_TIPO_VOTAVEL == tipo_voto]
    if cargo is not None:
        r = r[r.DS_CARGO_PERGUNTA == cargo]
    if r.empty:
        return {"error": "nenhum voto encontrado com esses filtros"}
    return {
        "total_votos": int(r["QT_VOTOS_N"].sum()),
        "tipo_voto": tipo_voto,
        "cargo": cargo,
        "linhas": len(r),
    }


def total_sem_biometria(zona=None, municipio=None, **kwargs):
    """Total de eleitores sem biometria válida, opcionalmente por zona ou município."""
    r = bu.copy()
    r["ZONA_N"] = pd.to_numeric(r["ZONA_N"], errors="coerce")
    r["SECAO_N"] = pd.to_numeric(r["SECAO_N"], errors="coerce")
    if zona is not None:
        z, err = _parse_int(zona, "zona")
        if err:
            return err
        if z == 0:
            return {"error": "zona 0 não existe. Omita o parâmetro para todas as zonas."}
        r = r[r["ZONA_N"] == z]
    if municipio is not None:
        r = r[r.NM_MUNICIPIO.str.upper() == str(municipio).upper()]
    if r.empty:
        return {"error": "nenhuma seção encontrada com esses filtros"}

    por_secao = r.drop_duplicates(subset=["ZONA_N", "SECAO_N"])
    biom = pd.to_numeric(por_secao["QT_ELEI_BIOM_SEM_HABILITACAO"], errors="coerce").fillna(0)
    aptos_s = pd.to_numeric(por_secao["QT_APTOS"], errors="coerce").fillna(0)
    total = int(biom.sum())
    aptos = int(aptos_s.sum())
    return {
        "total_sem_biometria": total,
        "total_aptos": aptos,
        "percentual": round(100 * total / aptos, 2) if aptos else None,
        "secoes": len(por_secao),
        "filtros": {"zona": zona, "municipio": municipio},
    }


def secao_details(zona, secao):
    """Todos os dados de uma seção."""
    z, err = _parse_int(zona, "zona")
    if err:
        return err
    s, err = _parse_int(secao, "secao")
    if err:
        return err

    r = bu.copy()
    r["ZONA_N"] = pd.to_numeric(r["ZONA_N"], errors="coerce")
    r["SECAO_N"] = pd.to_numeric(r["SECAO_N"], errors="coerce")
    r["QT_VOTOS_N"] = pd.to_numeric(r["QT_VOTOS"], errors="coerce").fillna(0)
    r = r[(r["ZONA_N"] == z) & (r["SECAO_N"] == s)]
    if r.empty:
        return {"error": f"seção {z}/{s} não encontrada"}

    primeira = r.iloc[0]
    votos = (
        r.groupby(["DS_CARGO_PERGUNTA", "DS_TIPO_VOTAVEL"], as_index=False)["QT_VOTOS_N"]
        .sum()
        .assign(QT_VOTOS=lambda d: d["QT_VOTOS_N"].astype(int))
        .drop(columns="QT_VOTOS_N")
        .to_dict(orient="records")
    )
    return {
        "zona": int(z),
        "secao": int(s),
        "municipio": primeira["NM_MUNICIPIO"],
        "aptos": int(_num(primeira["QT_APTOS"])),
        "comparecimento": int(_num(primeira["QT_COMPARECIMENTO"])),
        "abstencoes": int(_num(primeira["QT_ABSTENCOES"])),
        "sem_biometria": int(_num(primeira["QT_ELEI_BIOM_SEM_HABILITACAO"])),
        "votos": votos,
    }


def compare_secoes(zona_a, secao_a, zona_b, secao_b, tipo_voto=None):
    """Compara duas seções lado a lado, agregando por cargo."""
    if tipo_voto is not None and tipo_voto not in ("Nulo", "Branco", "Nominal"):
        return {"error": f"tipo_voto '{tipo_voto}' inválido."}

    ids = []
    for nome, valor in (("zona_a", zona_a), ("secao_a", secao_a),
                        ("zona_b", zona_b), ("secao_b", secao_b)):
        v, err = _parse_int(valor, nome)
        if err:
            return err
        ids.append(v)

    base = bu.copy()
    base["ZONA_N"] = pd.to_numeric(base["ZONA_N"], errors="coerce")
    base["SECAO_N"] = pd.to_numeric(base["SECAO_N"], errors="coerce")
    base["QT_VOTOS_N"] = pd.to_numeric(base["QT_VOTOS"], errors="coerce").fillna(0)
    base["AGREGADA"] = base["DS_SECOES_AGREGADAS"].astype(str).ne("#NULO#")

    secoes = []
    for i in range(0, 4, 2):
        z, s = ids[i], ids[i + 1]
        r_full = base[(base["ZONA_N"] == z) & (base["SECAO_N"] == s)]
        if r_full.empty:
            return {"error": f"seção {z}/{s} não encontrada"}
        r = r_full if tipo_voto is None else r_full[r_full.DS_TIPO_VOTAVEL == tipo_voto]
        primeira = r_full.iloc[0]
        votos_cargo = r.groupby("DS_CARGO_PERGUNTA")["QT_VOTOS_N"].sum().to_dict()
        secoes.append({
            "chave": f"{z}/{s}",
            "NR_ZONA": int(primeira["ZONA_N"]),
            "NR_SECAO": int(primeira["SECAO_N"]),
            "NM_MUNICIPIO": primeira["NM_MUNICIPIO"],
            "QT_APTOS": int(_num(primeira["QT_APTOS"])),
            "QT_COMPARECIMENTO": int(_num(primeira["QT_COMPARECIMENTO"])),
            "QT_ABSTENCOES": int(_num(primeira["QT_ABSTENCOES"])),
            "votos_por_cargo": {k: int(v) for k, v in votos_cargo.items()},
            "total_votos": int(r["QT_VOTOS_N"].sum()),
            "AGREGADA": bool(primeira["AGREGADA"]),
            "DS_SECOES_AGREGADAS": primeira["DS_SECOES_AGREGADAS"],
        })

    a, b = secoes
    best = {
        "mais_aptos": [a["chave"] if a["QT_APTOS"] >= b["QT_APTOS"] else b["chave"]],
        "mais_comparecimento": [a["chave"] if a["QT_COMPARECIMENTO"] >= b["QT_COMPARECIMENTO"] else b["chave"]],
        "mais_abstencoes": [a["chave"] if a["QT_ABSTENCOES"] >= b["QT_ABSTENCOES"] else b["chave"]],
        "mais_votos": [a["chave"] if a["total_votos"] >= b["total_votos"] else b["chave"]],
    }
    vencedor = best["mais_votos"][0]
    resposta = (
        f"{a['chave']} tem mais votos: {a['total_votos']} vs {b['total_votos']}"
        if vencedor == a["chave"]
        else f"{b['chave']} tem mais votos: {b['total_votos']} vs {a['total_votos']}"
    )
    resultado = {
        "resposta": resposta,
        "secoes": secoes,
        "best": best,
        "tipo_voto_aplicado": tipo_voto,
    }
    agregadas = [s["chave"] for s in secoes if s["AGREGADA"]]
    if agregadas:
        resultado["aviso"] = (
            f"As seções {', '.join(agregadas)} são CONSOLIDADAS. "
            "QT_APTOS, QT_COMPARECIMENTO e QT_ABSTENCOES não são comparáveis "
            "entre elas e uma seção individual."
        )
    return resultado


def search_secoes(tipo_voto=None, zona=None, cargo=None, min_votos=None, top_n=5):
    """Busca seções no BU por tipo de voto, zona, cargo e mínimo de votos."""
    TIPOS = ("Nulo", "Branco", "Nominal")
    CARGOS = ("Governador", "Presidente")
    if cargo in TIPOS:
        if tipo_voto is None:
            tipo_voto = cargo
        cargo = None
    if cargo is not None and cargo not in CARGOS:
        return {"error": f"cargo '{cargo}' inválido. Use 'Governador' ou 'Presidente'."}
    if tipo_voto is not None and tipo_voto not in TIPOS:
        return {"error": f"tipo_voto '{tipo_voto}' inválido. Use 'Nulo', 'Branco' ou 'Nominal'."}

    r = bu.copy()
    r["ZONA_N"] = pd.to_numeric(r["ZONA_N"], errors="coerce")
    r["SECAO_N"] = pd.to_numeric(r["SECAO_N"], errors="coerce")
    r["QT_VOTOS_N"] = pd.to_numeric(r["QT_VOTOS"], errors="coerce").fillna(0)
    if tipo_voto is not None:
        r = r[r.DS_TIPO_VOTAVEL == tipo_voto]
    if zona is not None:
        zona_n, err = _parse_int(zona, "zona")
        if err:
            return err
        if zona_n == 0:
            return {"error": "zona 0 não existe. Omita o parâmetro zona para buscar em todas as zonas."}
        r = r[r["ZONA_N"] == zona_n]
    if cargo is not None:
        r = r[r.DS_CARGO_PERGUNTA == cargo]
    if min_votos is not None:
        mv, err = _parse_int(min_votos, "min_votos")
        if err:
            return err
        r = r[r["QT_VOTOS_N"] >= mv]
    if r.empty:
        return {"error": "nenhuma seção encontrada com esses filtros"}

    agg = (
        r.groupby(["ZONA_N", "SECAO_N", "NM_MUNICIPIO"], as_index=False)["QT_VOTOS_N"]
        .sum()
        .sort_values("QT_VOTOS_N", ascending=False)
    )
    top = agg.head(int(top_n or 5))
    return {
        "total_found": len(agg),
        "resumo": f"{len(agg)} seções atendem ao filtro. Abaixo, as {len(top)} com mais votos.",
        "filtros_aplicados": {"tipo_voto": tipo_voto, "zona": zona,
                              "cargo": cargo, "min_votos": min_votos},
        "secoes": top.to_dict(orient="records"),
    }


FUNCTIONS = {
    "search_secoes": search_secoes,
    "secao_details": secao_details,
    "compare_secoes": compare_secoes,
    "total_por_tipo": total_por_tipo,
    "total_sem_biometria": total_sem_biometria,
}

# Célula 5 — Parte 3: a descrição das ferramentas

In [72]:
# @title 📋 Parte 3b · TOOLS completo (schemas limpos)

TOOLS = [
    {"type": "function",
     "function": {
         "name": "search_secoes",
         "description": ("Busca seções no Boletim de Urna por tipo de voto "
                         "(Nulo, Branco, Nominal), zona eleitoral, cargo "
                         "(Governador ou Presidente) e mínimo de votos. "
                         "Devolve quantas seções bateram no filtro e as 5 com "
                         "mais votos."),
         "parameters": {
             "type": "object",
             "properties": {
                 "tipo_voto": {
                     "type": "string",
                     "enum": ["Nulo", "Branco", "Nominal"],
                     "description": ("Tipo de voto a filtrar. OMITA para buscar "
                                     "todos os tipos.")
                 },
                 "zona": {
                     "type": "integer",
                     "description": ("Número da zona eleitoral. OMITA para buscar "
                                     "em todas as zonas. NÃO passe 0.")
                 },
                 "cargo": {
                     "type": "string",
                     "enum": ["Governador", "Presidente"],
                     "description": ("Cargo a filtrar. NÃO use 'Nulo', 'Branco' "
                                     "ou 'Nominal' aqui. OMITA para todos os cargos.")
                 },
                 "min_votos": {
                     "type": "integer",
                     "description": "Mínimo de votos na seção"
                 },
                 "top_n": {
                     "type": "integer",
                     "description": "Quantas seções devolver (padrão 5)"
                 },
             },
             "required": [],
         }}},
    {"type": "function",
     "function": {
         "name": "total_sem_biometria",
         "description": ("Total de eleitores sem biometria válida no arquivo, "
                         "opcionalmente filtrado por zona ou município. "
                         "Devolve também o percentual sobre os aptos. "
                         "Aceita APENAS zona e municipio — NÃO aceita tipo_voto, "
                         "cargo, min_votos nem top_n."),
         "parameters": {
             "type": "object",
             "properties": {
                 "zona": {
                     "type": "integer",
                     "description": "Número da zona. OMITA para todas as zonas."
                 },
                 "municipio": {
                     "type": "string",
                     "description": "Nome do município. OMITA para todos."
                 },
             },
             "required": [],
        }}},

    {"type": "function",
     "function": {
         "name": "secao_details",
         "description": ("Todos os dados de uma seção: aptos, comparecimento, "
                         "abstenções, horários, eleitores sem biometria, e os "
                         "votos agregados por cargo e tipo de voto."),
         "parameters": {
             "type": "object",
             "properties": {
                 "zona": {"type": "integer", "description": "Número da zona"},
                 "secao": {"type": "integer", "description": "Número da seção"},
             },
             "required": ["zona", "secao"],
         }}},

    {"type": "function",
     "function": {
         "name": "compare_secoes",
         "description": ("Compara duas seções lado a lado (aptos, "
                         "comparecimento, abstenções, votos por cargo). "
                         "Devolve o total de votos de cada seção e o vencedor "
                         "de cada critério."),
         "parameters": {
             "type": "object",
             "properties": {
                 "zona_a": {"type": "integer"},
                 "secao_a": {"type": "integer"},
                 "zona_b": {"type": "integer"},
                 "secao_b": {"type": "integer"},
                 "tipo_voto": {
                     "type": "string",
                     "enum": ["Nulo", "Branco", "Nominal"],
                     "description": ("Tipo de voto a comparar. OMITA para somar "
                                     "todos os tipos.")
                 },
             },
             "required": ["zona_a", "secao_a", "zona_b", "secao_b"],
         }}},

    {"type": "function",
      "function": {
          "name": "total_por_tipo",
          "description": ("Total de votos no arquivo inteiro, opcionalmente "
                          "filtrado por tipo de voto e/ou cargo. Use quando o "
                          "usuário pedir um total geral, não uma lista de seções."),
          "parameters": {
              "type": "object",
              "properties": {
                  "tipo_voto": {
                      "type": "string",
                      "enum": ["Nulo", "Branco", "Nominal"],
                      "description": "OMITA para somar todos os tipos de voto."
                  },
                  "cargo": {
                      "type": "string",
                      "enum": ["Governador", "Presidente"],
                      "description": "OMITA para somar os dois cargos."
                  },
              },
              "required": [],
          }}},
]

FUNCTIONS = {
    "search_secoes": search_secoes,
    "secao_details": secao_details,
    "compare_secoes": compare_secoes,
    "total_por_tipo": total_por_tipo,
    "total_sem_biometria": total_sem_biometria,
}

# Célula 6 — Parte 4: o system prompt

In [73]:
SYSTEM = f"""Agente de auditoria do BU das Eleições {ANO} da UF {UF}, turno {TURNO}.

FERRAMENTAS:
- search_secoes: busca seções e devolve o TOP N por votos. Parâmetros: tipo_voto, cargo, zona, min_votos, top_n.
- secao_details: dados completos de uma seção (aptos, comparecimento, abstenções, biometria, votos por cargo/tipo).
- compare_secoes: compara duas seções e identifica qual tem mais aptos, comparecimento, abstenções e votos.
- total_por_tipo: TOTAL de votos no arquivo. Parâmetros: tipo_voto, cargo.
- total_sem_biometria: total de eleitores sem biometria válida. Parâmetros: zona, municipio.

ROTEAMENTO:
- "quantos", "total", "somando", "no total" → total_por_tipo
- "quais seções", "onde", "top", "mais votos" → search_secoes
- "detalhes da seção X" → secao_details
- "compara X com Y" → compare_secoes
- "sem biometria" → total_sem_biometria
- NUNCA use search_secoes para responder "quantos" — ela devolve o top por seção, não o total.

PARÂMETROS:
- cargo aceita APENAS "Governador" ou "Presidente".
- tipo_voto aceita APENAS "Nulo", "Branco" ou "Nominal".
- NUNCA passe "Nulo", "Branco" ou "Nominal" em cargo.

REGRAS:
1. Se o usuário não citou um filtro, NÃO passe esse parâmetro.
   "no total" ou "no arquivo" significa sem filtro.
2. Se a ferramenta devolver "error", corrija os parâmetros e chame
   de novo. NÃO responda ao usuário com o erro.
   Se o erro for de parâmetro inválido, leia a mensagem e ajuste:
   "Nulo", "Branco" e "Nominal" são tipo_voto, nunca cargo.
   Se o erro disser que um parâmetro não existe, remova-o e chame de novo.
3. Quando a ferramenta devolver dados, responda com os dados e
   ENCERRE. Não acrescente frases de encerramento, ressalvas ou
   avisos ao final.
4. FORA DO ESCOPO: o agente só responde sobre o BU (votos por seção,
   zona, cargo e tipo). Perguntas sobre pesquisas eleitorais, nomes
   de candidatos (Lula, Bolsonaro, qualquer político), geografia,
   história, conhecimento geral — NÃO responda com conhecimento
   próprio. Diga apenas que o dado não está no arquivo e pare.
   Exemplo: "Qual é a capital do RN?" → "Esse dado não está no arquivo."
5. Em compare_secoes, use o campo "resposta" como está, citando os
   DOIS números. NÃO recalcule.
6. Nunca invente números. Cite zona e seção.
7. Ao citar números, copie exatamente como aparecem na observação da ferramenta.
   Nunca reformate: 4.264.428 não é 426.4428. 55.812 não é 55,812.
8. Ao listar seções, escreva "Zona X, Seção Y" — nunca os nomes de coluna ZONA_N ou SECAO_N.
9. Ao comparar seções, cite SEMPRE os dois totais, no formato "X tem mais: A vs B".
   Nunca omita o número da seção perdedora.
10. Ao citar percentuais, copie exatamente: 6,8% — nunca 6.8%.
11. Ao citar municípios, preserve o nome exatamente como aparece na observação.

Responda em português, direto."""

PEDIDO = "Quais seções têm mais votos nulos para presidente?"
print("system prompt definido")

system prompt definido


# Célula 7 — Parte 5: o loop completo

In [74]:
# @title 🔁 Parte 5b · summary completo (todas as ferramentas)

def summary(result):
    if not isinstance(result, dict):
        return str(result)

    if "error" in result:
        return f'❌ erro: {result["error"]}'

    if "total_found" in result:
        secoes = [f'{s["ZONA_N"]}/{s["SECAO_N"]} ({s["NM_MUNICIPIO"]}): {s["QT_VOTOS_N"]}'
                  for s in result.get("secoes", [])]
        return f'{result["total_found"]} seções. Top: ' + " | ".join(secoes)

    if "total_sem_biometria" in result:
        pct = result.get("percentual")
        pct_txt = f" ({str(pct).replace('.', ',')}% dos aptos)" if pct is not None else ""
        total = f'{result["total_sem_biometria"]:,}'.replace(",", ".")
        return f'{total} sem biometria{pct_txt} em {result["secoes"]} seções'

    if "total_votos" in result and "secoes" not in result:
        filtro = result.get("tipo_voto") or "todos os tipos"
        cargo = result.get("cargo")
        if cargo:
            filtro = f"{filtro} para {cargo}"
        linhas = result.get("linhas")
        total = f'{result["total_votos"]:,}'.replace(",", ".")
        base = f'{total} votos ({filtro})'
        return f'{base} em {linhas} linhas' if linhas is not None else base

    if "resposta" in result:
        return result["resposta"]

    if "votos" in result:
        por_cargo = {}
        for v in result["votos"]:
            por_cargo.setdefault(v["DS_CARGO_PERGUNTA"], 0)
            por_cargo[v["DS_CARGO_PERGUNTA"]] += v["QT_VOTOS"]
        totais = " | ".join(f'{k}: {v}' for k, v in por_cargo.items())
        return (f'Seção {result["zona"]}/{result["secao"]} — {result["municipio"]}: '
                f'{result["aptos"]} aptos, {result["comparecimento"]} comparecimento. '
                f'Votos por cargo: {totais}')

    return str(result)[:300]


def run_agent(pedido, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": pedido}]
    print(f"👤 {pedido}\n")
    for volta in range(1, max_steps + 1):
        print(f"── volta {volta} " + "─" * 60)
        msg = llm(messages, tools=TOOLS)["message"]
        messages.append(msg)
        if not msg.get("tool_calls"):
            print("🤖 o modelo respondeu com texto → fim do loop\n")
            print(msg["content"])
            return messages
        for c in msg["tool_calls"]:
            name = c["function"]["name"]
            raw = c["function"]["arguments"]
            args = raw if isinstance(raw, dict) else json.loads(raw or "{}")
            print(f"🤖 o modelo pediu     → {name}({json.dumps(args, ensure_ascii=False)})")
            result = FUNCTIONS[name](**args)
            print(f"🔧 nós executamos    → {summary(result)}")
            messages.append({"role": "tool", "tool_name": name,
                             "content": json.dumps(result, ensure_ascii=False)})
    print(f"(parei depois de {max_steps} voltas sem resposta final)")
    return messages


conversa = run_agent(PEDIDO)

👤 Quais seções têm mais votos nulos para presidente?

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu     → search_secoes({"cargo": "Presidente", "tipo_voto": "Nulo"})
🔧 nós executamos    → 8083 seções. Top: 25/105 (CAICÓ): 26 | 4/16 (NATAL): 23 | 4/426 (NATAL): 22 | 51/176 (SÃO GONÇALO DO AMARANTE): 22 | 69/281 (NATAL): 22
── volta 2 ────────────────────────────────────────────────────────────
🤖 o modelo respondeu com texto → fim do loop

As seções com mais votos nulos para presidente são as seguintes:

- Zona 25, Seção 105, município de Caicó, com 26 votos.
- Zona 4, Seção 16, município de Natal, com 23 votos.
- Zona 4, Seção 426, município de Natal, com 22 votos.
- Zona 51, Seção 176, município de São Gonçalo do Amarante, com 22 votos.
- Zona 69, Seção 281, município de Natal, com 22 votos.


In [75]:
# @title 🧪 Parte 6b · Bateria final

TESTES = [
    {"pergunta": "Quais seções têm mais votos nulos para presidente?",
     "esperado": "Caicó, zona 25 seção 105, com 26 votos nulos",
     "tipo": "resposta", "checar": ["25", "105", "26"],
     "proibido": ["não existe", "não encontrad"]},

   {"pergunta": "Quais seções têm mais votos nulos para governador?",
    "esperado": "Parelhas, zona 24 seção 35, com 37 votos nulos",
    "tipo": "resposta",
    "checar": ["24", "35", "37"],
    "proibido": ["não existe", "não encontrada"]},

    {"pergunta": "Me dá os detalhes da seção 505 da zona 3.",
     "esperado": "Seção existe, com votos de Governador e Presidente",
     "tipo": "resposta", "checar": ["505", "governador", "presidente"],
     "proibido": ["não existe", "não encontrad"]},

    {"pergunta": "Compara os votos nominais da seção 505 com a 504 da zona 3. Qual tem mais?",
     "esperado": "505 tem mais (838 vs 834 nominais)",
     "tipo": "resposta",
     "checar": ["505", "838"],
     "proibido": ["não existe", "não encontrada", "504 tem mais"]},

    {"pergunta": "Qual o total de votos nominais no arquivo?",
     "esperado": "4.264.428",
     "tipo": "resposta", "checar": ["4.264.428", "4264428"],
     "proibido": ["não existe"]},

    {"pergunta": "Quantos votos brancos existem no total?",
     "esperado": "55.812",
     "tipo": "resposta", "checar": ["55.812", "55812"],
     "proibido": ["não existe"]},

    {"pergunta": "Qual foi a intenção de voto para governador nas pesquisas?",
     "esperado": "RECUSA — dado não existe no arquivo",
     "tipo": "recusa",
     "checar": ["não existe", "não tenho", "não posso", "não fornece"]},

    {"pergunta": "Quantos votos o candidato Lula teve?",
     "esperado": "RECUSA — dado não existe no arquivo",
     "tipo": "recusa",
     "checar": ["não existe", "não tenho", "não posso"]},

    {"pergunta": "Qual é a capital do Rio Grande do Norte?",
     "esperado": "RECUSA — conhecimento externo, fora do escopo",
     "tipo": "recusa",
     "checar": ["não existe", "não posso", "fora do escopo",
                "só respondo", "apenas sobre"]},
    {"pergunta": "Quantos eleitores estão sem biometria válida no total?",
     "esperado": "180.838 sem biometria (6,8% dos aptos)",
     "tipo": "resposta",
     "checar": ["180.838", "6,8"],
     "proibido": ["não existe", "não está"]},

    {"pergunta": "Quais seções têm mais votos nulos?",
     "esperado": "Carnaúba dos Dantas, zona 22 seção 41, com 52 votos nulos",
     "tipo": "resposta",
     "checar": ["22", "41", "52"],
     "proibido": ["não existe", "não encontrada"]},
]


def extrair_resposta(conversa):
    """Pega a última mensagem de texto do assistente."""
    for m in reversed(conversa or []):
        if m.get("role") == "assistant" and m.get("content"):
            return m["content"]
    return ""


def avaliar(teste, resposta):
    texto = (resposta or "").lower()
    texto_num = texto.replace(".", "").replace(",", "")

    for p in teste.get("proibido", []):
        if p.lower() in texto:
            return False, [f"contém proibido: '{p}'"]

    if teste["tipo"] == "resposta":
        faltando = []
        for c in teste["checar"]:
            c_norm = c.lower().replace(".", "").replace(",", "")
            if c.lower() not in texto and c_norm not in texto_num:
                faltando.append(c)
        return len(faltando) == 0, faltando

    # recusa: aceita variações naturais
    gatilhos = [
        "não existe", "não está", "não tenho", "não posso",
        "não fornece", "não disponível", "não consta",
        "fora do escopo", "não faz parte", "não há",
        "só respondo", "apenas sobre", "não contempla",
    ]
    achou = any(g in texto for g in gatilhos)
    return achou, [] if achou else teste["checar"]


resultados = []
for i, t in enumerate(TESTES, 1):
    print(f"\n{'═' * 72}")
    print(f"TESTE {i} [{t['tipo'].upper()}]")
    print(f"Esperado: {t['esperado']}")
    print(f"{'═' * 72}")

    conversa = run_agent(t["pergunta"])
    resposta = extrair_resposta(conversa)
    ok, faltando = avaliar(t, resposta)
    resultados.append({"teste": i, "ok": ok, "faltando": faltando,
                       "resposta": resposta[:200]})
    print(f"\n{'✅ PASSOU' if ok else '❌ FALHOU'}"
          + (f" — {faltando}" if faltando else ""))

passou = sum(1 for r in resultados if r["ok"])
print(f"\n{'═' * 72}")
print(f"PLACAR: {passou}/{len(TESTES)} testes passaram")
print(f"{'═' * 72}")
for r in resultados:
    print(f"  Teste {r['teste']}: {'✅' if r['ok'] else '❌'}"
          + (f" — {r['faltando']}" if r['faltando'] else ""))


════════════════════════════════════════════════════════════════════════
TESTE 1 [RESPOSTA]
Esperado: Caicó, zona 25 seção 105, com 26 votos nulos
════════════════════════════════════════════════════════════════════════
👤 Quais seções têm mais votos nulos para presidente?

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu     → search_secoes({"tipo_voto": "Nulo", "cargo": "Presidente"})
🔧 nós executamos    → 8083 seções. Top: 25/105 (CAICÓ): 26 | 4/16 (NATAL): 23 | 4/426 (NATAL): 22 | 51/176 (SÃO GONÇALO DO AMARANTE): 22 | 69/281 (NATAL): 22
── volta 2 ────────────────────────────────────────────────────────────
🤖 o modelo respondeu com texto → fim do loop

As seções com mais votos nulos para presidente são:

- Zona 25, Seção 105, município de Caicó, com 26 votos.
- Zona 4, Seção 16, município de Natal, com 23 votos.
- Zona 4, Seção 426, município de Natal, com 22 votos.
- Zona 51, Seção 176, município de São Gonçalo do Amarante, com 22 votos.
- 

In [ ]:
# @title 💬 Perguntas livres em loop

print("Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("👤 ")
    if pergunta.strip().lower() in ("sair", "exit", "q"):
        break
    if not pergunta.strip():
        continue

    run_agent(pergunta)      # ← sem print aqui